In [1]:
! pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 63.3 MB/s eta 0:00:00


In [2]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tqdm
from glob import glob
from scipy import ndimage
from einops import rearrange
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc as sklearn_auc
 
from monai.networks.nets.masked_autoencoder_vit import MaskedAutoEncoderViT
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, ResizeWithPadOrCropd, EnsureTyped,
    RandFlipd, RandAffined, RandGaussianNoised, RandAdjustContrastd,
    SpatialPadd, RandSpatialCropd,
)
from monai.data import CacheDataset, DataLoader, Dataset
from monai.losses import SSIMLoss


## Parameters
IMG_SIZE = 96
PATCH_SIZE = 8
MASKING_RATIO = 0.85

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-05-14 13:02:20.721226: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778763740.931504      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778763740.992584      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778763741.504922      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778763741.504974      22 computation_placer.cc:1

In [3]:
# =============================================================================
# 1. DATA LOADING  — paste your existing loading code here
# =============================================================================
data_root = '/kaggle/input/datasets/almadavidson1/fcd-database'
tsv_path  = os.path.join(data_root, "participants.tsv")
df        = pd.read_csv(tsv_path, sep='\t')
 
data_dicts = []
for idx, row in df.iterrows():
    p_id = row['participant_id']
 
    image_pattern = os.path.join(data_root, 'DATA', p_id, 'anat', f"*_T1w.nii*", f"*_T1w.nii*")
    img_files     = glob(image_pattern)
 
    mask_pattern  = os.path.join(data_root, 'DATA', p_id, 'anat', f"*-FLAIR_roi_inT1.nii*")
    mask_files   = glob(mask_pattern)
 
    if img_files:
        entry = {"image": img_files[0], "group": row.get('group', 'hc')}
        if mask_files:
            entry["mask"] = mask_files[0]
        data_dicts.append(entry)
 
hc_dict       = [d for d in data_dicts if str(d['group']).lower() == 'hc']
fcd_dict      = [d for d in data_dicts if str(d['group']).lower() == 'fcd' and 'mask' in d]
fcd_dict_clean = [d for d in fcd_dict if os.path.getsize(d["mask"]) > 0]
 
print(f"HC subjects:             {len(hc_dict)}")
print(f"FCD subjects with mask:  {len(fcd_dict_clean)}")

val_hc      = hc_dict[75:]

HC subjects:             85
FCD subjects with mask:  85


In [4]:
# Data transform and load FCD data and healthy data
hc_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    EnsureTyped(keys=["image"], dtype=torch.float32),
])

fcd_transforms = Compose([
    LoadImaged(keys=["image","mask"]),
    EnsureChannelFirstd(keys=["image","mask"]),
    Orientationd(keys=["image","mask"], axcodes="RAS"),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    EnsureTyped(keys=["image","mask"], dtype=torch.float32),
])

hc_ds   = Dataset(data=val_hc, transform=hc_transforms)
fcd_ds   = Dataset(data=fcd_dict_clean, transform=fcd_transforms)

hc_loader   = DataLoader(hc_ds, batch_size=1, shuffle=False, num_workers=2)
fcd_loader   = DataLoader(fcd_ds, batch_size=1, shuffle=False, num_workers=2)

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [5]:
# =============================================================================
# 4. MODEL
# =============================================================================
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MaskedAutoEncoderViT(
    spatial_dims=3,
    in_channels=1,
    img_size=(IMG_SIZE, IMG_SIZE, IMG_SIZE),
    patch_size=(PATCH_SIZE, PATCH_SIZE, PATCH_SIZE),
    masking_ratio=MASKING_RATIO,
).to(DEVICE)

pre_weights_path = "/kaggle/input/datasets/almadavidson/pretrained-weights/mae_best_model_updated.pth"
checkpoint = torch.load(pre_weights_path, map_location=DEVICE)
model.load_state_dict(torch.load(pre_weights_path)['model_state_dict'])
print(f"  Loaded epoch {checkpoint['epoch']}  (val loss={checkpoint['val_loss']:.5f})")

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  Loaded epoch 1738  (val loss=0.09379)


In [6]:
# =============================================================================
# 1. INFERENCE UTILITIES
# =============================================================================

def mae_reconstruct_fullbrain(model, x, stride=48, n_runs=5):
    """
    Slide a 96³ window across the full brain with 50% overlap.
    At each window: run N MC reconstructions with different random masks.
    Average all overlapping reconstructions to get final result.

    MC     → ensures lesion patch gets masked at least once (P=0.9999 at masking=0.85)
    Sliding → eliminates patch boundary artifacts across full brain
    """
    model.eval()
    B, C, H, W, D = x.shape
    win = IMG_SIZE  

    # Pad so windows tile evenly
    def pad_to_fit(size):
        remainder = size % stride
        return 0 if remainder == 0 else stride - remainder

    ph = pad_to_fit(H); pw = pad_to_fit(W); pd = pad_to_fit(D)
    x_pad     = F.pad(x, (0, pd, 0, pw, 0, ph), mode='reflect')
    Ph, Pw, Pd = x_pad.shape[2:]

    acc_pad   = torch.zeros_like(x_pad)
    count_pad = torch.zeros_like(x_pad)

    std_acc   = torch.zeros_like(x_pad)
    std_count = torch.zeros_like(x_pad)

    # Window start positions
    h_starts = list(range(0, Ph - win + 1, stride))
    w_starts = list(range(0, Pw - win + 1, stride))
    d_starts = list(range(0, Pd - win + 1, stride))

    # Always include last window to avoid missing edges
    if not h_starts or h_starts[-1] + win < Ph: h_starts.append(max(0, Ph - win))
    if not w_starts or w_starts[-1] + win < Pw: w_starts.append(max(0, Pw - win))
    if not d_starts or d_starts[-1] + win < Pd: d_starts.append(max(0, Pd - win))

    total = len(h_starts) * len(w_starts) * len(d_starts)
    print(f"  Full brain sliding window: {total} windows "
          f"({len(h_starts)}×{len(w_starts)}×{len(d_starts)}) "
          f"× {n_runs} MC runs = {total * n_runs} forward passes")

    all_window_recons = {}   # store for std computation

    with torch.no_grad():
        for h0 in h_starts:
            for w0 in w_starts:
                for d0 in d_starts:
                    window = x_pad[:, :, h0:h0+win, w0:w0+win, d0:d0+win]

                    # MC: n_runs random masks, average
                    recons = []
                    for _ in range(n_runs):
                        pred, mask   = model(window)
                        target       = patchify(window)
                        comp_patches = pred * mask.unsqueeze(-1) + \
                                       target * (1 - mask.unsqueeze(-1))
                        recon        = unpatchify(comp_patches)
                        recons.append(recon)

                    recons_t = torch.stack(recons, dim=0)
                    recon_mean = recons_t.mean(dim=0).to(DEVICE)
                    recon_std  = recons_t.std(dim=0).to(DEVICE)
                    del recons, recons_t

                    acc_pad[:, :, h0:h0+win, w0:w0+win, d0:d0+win]   += recon_mean
                    count_pad[:, :, h0:h0+win, w0:w0+win, d0:d0+win] += 1.0

                    std_acc[:, :, h0:h0+win, w0:w0+win, d0:d0+win]   += recon_std
                    std_count[:, :, h0:h0+win, w0:w0+win, d0:d0+win] += 1.0
                    del recon_mean, recon_std

    # Average overlapping windows
    result_pad = acc_pad / count_pad.clamp(min=1)
    std_pad = std_acc / std_count.clamp(min=1)

    # Remove padding
    mean_recon = result_pad[:, :, :H, :W, :D]
    std_recon  = std_pad[:, :, :H, :W, :D]

    return mean_recon, std_recon


def compute_residual_map_v2(original, mean_recon, std_recon=None, smooth_sigma=1.5):
    """
    Residual map with brain tissue masking and normalization.
    Suppresses background, CSF/ventricles, and skull boundary artifacts.
    """
    orig_np  = original.squeeze().cpu().numpy()
    recon_np = mean_recon.squeeze().cpu().numpy()

    # L1 residual
    residual = np.abs(orig_np - recon_np)

    # Weight by uncertainty where available
    if std_recon is not None:
        std_np   = std_recon.squeeze().cpu().numpy()
        confidence_mask = std_np < np.percentile(std_np, 90)
        residual[~confidence_mask] = 0

    # Suppress background + CSF/ventricles (dark on z-scored T1)
    from skimage.filters import threshold_otsu
    nonzero_vals = orig_np[orig_np != 0]
    thresh = threshold_otsu(nonzero_vals)
    tissue_mask = orig_np > thresh
    residual[~tissue_mask] = 0

    # Erode brain mask to remove skull boundary artifacts
    eroded_mask = ndimage.binary_erosion(tissue_mask, iterations=7)
    residual[~eroded_mask] = 0

    # Normalize within brain tissue — makes subtle cortical differences visible
    brain_residual = residual[eroded_mask]
    if brain_residual.std() > 0:
        normalized = (brain_residual - brain_residual.mean()) / brain_residual.std()
        residual[eroded_mask] = np.clip(normalized, 0, None)

    if smooth_sigma > 0:
        residual = ndimage.gaussian_filter(residual, sigma=smooth_sigma)

    return residual, eroded_mask


def calibrate_threshold(model, hc_loader, n_runs=5, smooth_sigma=1.5):
    """Compute mean + std of residuals across healthy controls."""
    all_residuals = []
    for batch in hc_loader:
        volume = batch["image"].to(DEVICE)
        mean_recon, std_recon = mae_reconstruct_fullbrain(model, volume, n_runs=n_runs)
        residual, eroded_mask = compute_residual_map_v2(volume, mean_recon, std_recon, smooth_sigma=smooth_sigma)
        brain_values = residual[eroded_mask]
        all_residuals.append(brain_values)
    
    combined_voxels = np.concatenate(all_residuals)
    hc_mean = combined_voxels.mean()
    hc_std  = combined_voxels.std()
    
    return hc_mean, hc_std

def threshold_residual(residual, hc_mean, hc_std, z_score=2.0, min_size=10):
    """Flag voxels exceeding hc_mean + z_score * hc_std."""
    thresh = hc_mean + z_score * hc_std
    binary = residual > thresh

    labeled, n = ndimage.label(binary)
    for i in range(1, n + 1):
        if (labeled == i).sum() < min_size:
            binary[labeled == i] = False

    return binary


def run_anomaly_detection(model, volume, hc_mean, hc_std, 
                          z_score, n_runs=5, smooth_sigma=1.5):
    """Full pipeline: full brain volume → lesion mask + residual map."""
    mean_recon, std_recon = mae_reconstruct_fullbrain(
        model, volume, stride=48, n_runs=n_runs
    )
    print(mean_recon.shape)
    residual, brain_mask = compute_residual_map_v2(volume, mean_recon, std_recon, smooth_sigma=smooth_sigma)
    mask = threshold_residual(residual, hc_mean, hc_std, z_score=z_score)
    print(mask.shape)

    return {
        "original":       volume.squeeze().cpu().numpy(),
        "reconstruction": mean_recon.squeeze().cpu().numpy(),
        "uncertainty":    std_recon.squeeze().cpu().numpy(),
        "residual":       residual,
        "lesion_mask":    mask,
    }


In [7]:
# =============================================================================
# 2. EVALUATION ON FCD CASES
# =============================================================================

GRID = IMG_SIZE // PATCH_SIZE   # 6

def patchify(x):
    """(B, 1, H, W, D) → (B, N_patches, patch_volume)"""
    return rearrange(
        x, 'b c (h p1) (w p2) (d p3) -> b (h w d) (p1 p2 p3 c)',
        p1=PATCH_SIZE, p2=PATCH_SIZE, p3=PATCH_SIZE
    )
 
def unpatchify(patches):
    """(B, N_patches, patch_volume) → (B, 1, H, W, D)"""
    return rearrange(
        patches,
        'b (h w d) (p1 p2 p3 c) -> b c (h p1) (w p2) (d p3)',
        h=GRID, w=GRID, d=GRID,
        p1=PATCH_SIZE, p2=PATCH_SIZE, p3=PATCH_SIZE, c=1
    )


def dice_score(pred_mask, gt_mask):
    pred = pred_mask.astype(bool)
    gt   = gt_mask.astype(bool)
    intersection = (pred & gt).sum()
    if pred.sum() + gt.sum() == 0:
        return 1.0
    return 2 * intersection / (pred.sum() + gt.sum())
 

def evaluate_hc_baseline(model, val_loader, hc_mean, hc_std, z_score=2.0, n_runs=5):
    print("\n" + "="*60)
    print("HC BASELINE — FALSE POSITIVE CHECK")
    print("="*60)

    fp_counts   = []
    hc_scores   = [] 

    for i, batch in enumerate(tqdm.tqdm(val_loader, desc="Evaluating HC cases")):
        volume = batch["image"].to(DEVICE)

        results = run_anomaly_detection(
            model, volume,
            hc_mean=hc_mean, hc_std=hc_std,
            z_score=z_score, n_runs=n_runs,
            smooth_sigma=1.0,
        )

        fp_voxels = results["lesion_mask"].sum()
        fp_counts.append(fp_voxels)

        # 95th percentile residual as subject-level score
        subject_score = float(np.percentile(results["residual"], 95))
        hc_scores.append(subject_score)

        print(f"  HC {i+1:02d}: {fp_voxels} false positive voxels | score={subject_score:.4f}")

        del results
        torch.cuda.empty_cache()

    print(f"\nMean FP voxels across HC: {np.mean(fp_counts):.1f}")
    return fp_counts, hc_scores


def evaluate_fcd(model, fcd_loader, hc_mean, hc_std, hc_scores,
                 z_score=2.0, n_runs=10):
    print("\n" + "="*60)
    print("FCD EVALUATION")
    print("="*60)

    dice_scores  = []
    fcd_scores   = []
    os.makedirs("/tmp/residuals", exist_ok=True)

    for i, batch in enumerate(tqdm.tqdm(fcd_loader, desc="Evaluating FCD cases")):
        volume    = batch["image"].to(DEVICE)
        gt_mask   = batch["mask"].squeeze().cpu().numpy()
        gt_binary = (gt_mask > 0).astype(bool)

        results = run_anomaly_detection(
            model, volume,
            hc_mean=hc_mean, hc_std=hc_std,
            z_score=z_score, n_runs=n_runs,
            smooth_sigma=1.0,
        )

        # ── Dice ──────────────────────────────────────────────────
        d = dice_score(results["lesion_mask"], gt_binary)
        dice_scores.append(d)

        # ── Subject-level score ───────────────────────────────────
        subject_score = float(np.percentile(results["residual"], 95))
        fcd_scores.append(subject_score)

        # ── Save residuals to /tmp ──────
        np.save(f"/tmp/residuals/res_{i}.npy", results["residual"].flatten().astype(np.float32))
        np.save(f"/tmp/residuals/lbl_{i}.npy", gt_binary.flatten().astype(np.uint8))

        z_map = (results["residual"] - hc_mean) / hc_std
        print(f"  Case {i+1:02d}: Dice={d:.3f} | score={subject_score:.4f} | "
              f"max Z={z_map.max():.2f} | max Z@lesion={z_map[gt_binary].max():.2f}")

        # ── Per-case visualization ────────────────────────────────
        lesion_per_slice = results["lesion_mask"].sum(axis=(0, 1))
        s = lesion_per_slice.argmax() if lesion_per_slice.max() > 0 \
            else results["original"].shape[2] // 2

        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        axes[0].imshow(results["original"][:, :, s],       cmap="gray");  axes[0].set_title("Original (FCD)")
        axes[1].imshow(results["reconstruction"][:, :, s], cmap="gray");  axes[1].set_title("Reconstruction")
        axes[2].imshow(results["residual"][:, :, s],       cmap="hot");   axes[2].set_title("Residual map")
        axes[3].imshow(results["original"][:, :, s],       cmap="gray")
        axes[3].imshow(results["lesion_mask"][:, :, s],    cmap="Reds",   alpha=0.5)
        axes[3].imshow(gt_binary[:, :, s],                 cmap="Greens", alpha=0.3)
        axes[3].set_title(f"Pred (red) vs GT (green)\nDice={d:.3f}")
        for ax in axes: ax.axis("off")
        plt.tight_layout()
        plt.savefig(f"fcd_case_{i+1:02d}.png", dpi=100)
        plt.close()

        del results
        torch.cuda.empty_cache()

    # ── Subject-level ROC (FCD vs HC) ─────────────────────────────
    subject_scores = hc_scores + fcd_scores
    subject_labels = [0] * len(hc_scores) + [1] * len(fcd_scores)

    fpr, tpr, _ = roc_curve(subject_labels, subject_scores)
    roc_auc     = sklearn_auc(fpr, tpr)

    plt.figure(figsize=(7, 7))
    plt.plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate');  plt.ylabel('True Positive Rate (Sensitivity)')
    plt.title('Subject-level ROC: FCD vs Healthy Controls')
    plt.legend(loc="lower right");  plt.grid(alpha=0.3)
    plt.savefig("ROC_curve_subject_level.png", dpi=100)
    plt.close()

    mean_dice = np.mean(dice_scores)
    print(f"\nMean Dice: {mean_dice:.3f} ± {np.std(dice_scores):.3f}")
    print(f"Subject-level AUC: {roc_auc:.3f}")
    return dice_scores, roc_auc

In [8]:
# =============================================================================
# 3. RUN EVALUATION
# =============================================================================
 
hc_mean, hc_std = calibrate_threshold(model, hc_loader,smooth_sigma=1.0)

fp_counts, hc_scores = evaluate_hc_baseline(model, hc_loader, hc_mean, hc_std, z_score=2.0)

if len(fcd_ds) > 0:
    dice_scores, auc_val = evaluate_fcd(model, fcd_loader, hc_mean, hc_std, hc_scores, z_score=2.0)
else:
    print("No FCD cases with masks found — skipping evaluation.")
    print("To test, load a lesion volume manually:")
    print("  volume = fcd_transforms({'image': '/path/to/fcd.nii.gz', 'mask': '/path/to/mask.nii.gz'})")
    print("  results = run_anomaly_detection(model, volume['image'].unsqueeze(0).to(DEVICE))")


  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes

HC BASELINE — FALSE POSITIVE CHECK


Evaluating HC cases:   0%|          | 0/10 [00:00<?, ?it/s]

  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
torch.Size([1, 1, 208, 320, 320])
(208, 320, 320)


Evaluating HC cases:  10%|█         | 1/10 [01:22<12:21, 82.36s/it]

  HC 01: 83518 false positive voxels | score=0.1832
  Full brain sliding window: 144 windows (4×6×6) × 5 MC runs = 720 forward passes
torch.Size([1, 1, 208, 320, 320])
(208, 320, 320)


...

  Full brain sliding window: 144 windows (4×6×6) × 10 MC runs = 1440 forward passes
torch.Size([1, 1, 208, 320, 320])
(208, 320, 320)
  Case 84: Dice=0.003 | score=0.1154 | max Z=8.69 | max Z@lesion=4.73


Evaluating FCD cases:  99%|█████████▉| 84/85 [2:03:28<01:47, 107.23s/it]

  Full brain sliding window: 144 windows (4×6×6) × 10 MC runs = 1440 forward passes
torch.Size([1, 1, 208, 320, 320])
(208, 320, 320)
  Case 85: Dice=0.000 | score=0.1248 | max Z=18.06 | max Z@lesion=-0.57


Evaluating FCD cases: 100%|██████████| 85/85 [2:05:33<00:00, 88.63s/it] 



Mean Dice: 0.003 ± 0.008
Subject-level AUC: 0.242
